In [1]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("RetailIntelligence-TP4-Analyses-Metier")
    .master("local[*]").config("spark.hadoop.fs.defaultFS", "file:///")
    .getOrCreate()
)

DATA_ROOT = (
    Path.home()
    / "retail-intelligence"
    / "tp3"
    / "data"
)

store_sales_path = DATA_ROOT / "silver" / "store_sales"

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/21 00:50:35 WARN Utils: Your hostname, NAVID-DELL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/21 00:50:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/navid/retail-intelligence/tp4/retail_intel_tp4_venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/21 00:50:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# --------------------------------------------------
# ANALYSE - PRODUITS LES PLUS VENDUS PAR MAGASIN
# --------------------------------------------------

df_sales = spark.read.parquet(str(store_sales_path))

print("=== STRUCTURE STORE_SALES ===")
df_sales.printSchema()

# Vérification des colonnes nécessaires
colonnes_requises = [
    "ss_store_sk",
    "ss_item_sk",
    "ss_quantity",
    "ss_net_paid_inc_tax"
]

colonnes_absentes = [
    c for c in colonnes_requises
    if c not in df_sales.columns
]

if colonnes_absentes:
    raise ValueError(
        f"Colonnes absentes de store_sales : {colonnes_absentes}"
    )

# Vérification des valeurs manquantes
print("\n=== CONTRÔLE DES VALEURS MANQUANTES ===")

df_sales.select(
    *[
        F.sum(F.col(c).isNull().cast("long")).alias(c + "_nulls")
        for c in colonnes_requises
    ]
).show(truncate=False)

# Agrégation par magasin et produit
print("\n=== VENTES AGRÉGÉES PAR MAGASIN ET PRODUIT ===")

ventes_produits = (
    df_sales
    .filter(
        F.col("ss_store_sk").isNotNull()
        & F.col("ss_item_sk").isNotNull()
    )
    .groupBy("ss_store_sk", "ss_item_sk")
    .agg(
        F.sum("ss_quantity").alias("quantites_vendues"),
        F.sum("ss_net_paid_inc_tax").alias("ca_ttc"),
        F.count("*").alias("nombre_lignes_vente")
    )
)

# Classement des produits dans chaque magasin
fenetre_magasin = (
    Window
    .partitionBy("ss_store_sk")
    .orderBy(
        F.desc("quantites_vendues"),
        F.desc("ca_ttc")
    )
)

classement_produits = (
    ventes_produits
    .withColumn(
        "rang_produit",
        F.row_number().over(fenetre_magasin)
    )
)

# Afficher les 10 produits les plus vendus par magasin
top_produits = (
    classement_produits
    .filter(F.col("rang_produit") <= 10)
    .orderBy("ss_store_sk", "rang_produit")
)

print("\n=== TOP 10 DES PRODUITS PAR MAGASIN ===")
top_produits.show(120, truncate=False)

# Top 10 global, tous magasins confondus
print("\n=== TOP 10 GLOBAL DES PRODUITS ===")

top_global = (
    df_sales
    .filter(F.col("ss_item_sk").isNotNull())
    .groupBy("ss_item_sk")
    .agg(
        F.sum("ss_quantity").alias("quantites_vendues"),
        F.sum("ss_net_paid_inc_tax").alias("ca_ttc")
    )
    .orderBy(F.desc("quantites_vendues"))
    .limit(10)
)

top_global.show(truncate=False)

=== STRUCTURE STORE_SALES ===
root
 |-- ss_sold_date_sk: long (nullable = true)
 |-- ss_sold_time_sk: long (nullable = true)
 |-- ss_item_sk: long (nullable = true)
 |-- ss_customer_sk: long (nullable = true)
 |-- ss_cdemo_sk: long (nullable = true)
 |-- ss_hdemo_sk: long (nullable = true)
 |-- ss_addr_sk: long (nullable = true)
 |-- ss_store_sk: long (nullable = true)
 |-- ss_promo_sk: long (nullable = true)
 |-- ss_ticket_number: long (nullable = true)
 |-- ss_quantity: integer (nullable = true)
 |-- ss_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_list_price: decimal(7,2) (nullable = true)
 |-- ss_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_discount_amt: decimal(7,2) (nullable = true)
 |-- ss_ext_sales_price: decimal(7,2) (nullable = true)
 |-- ss_ext_wholesale_cost: decimal(7,2) (nullable = true)
 |-- ss_ext_list_price: decimal(7,2) (nullable = true)
 |-- ss_ext_tax: decimal(7,2) (nullable = true)
 |-- ss_coupon_amt: decimal(7,2) (nullable = true)
 |-- ss_net_p

+-----------------+----------------+-----------------+-------------------------+
|ss_store_sk_nulls|ss_item_sk_nulls|ss_quantity_nulls|ss_net_paid_inc_tax_nulls|
+-----------------+----------------+-----------------+-------------------------+
|0                |0               |0                |0                        |
+-----------------+----------------+-----------------+-------------------------+


=== VENTES AGRÉGÉES PAR MAGASIN ET PRODUIT ===

=== TOP 10 DES PRODUITS PAR MAGASIN ===


+-----------+----------+-----------------+----------+-------------------+------------+
|ss_store_sk|ss_item_sk|quantites_vendues|ca_ttc    |nombre_lignes_vente|rang_produit|
+-----------+----------+-----------------+----------+-------------------+------------+
|0          |10000     |39681            |1451905.71|781                |1           |
|0          |10003     |36943            |1349066.17|746                |2           |
|0          |10001     |35181            |1319938.74|696                |3           |
|0          |10002     |34129            |1186603.01|656                |4           |
|0          |5618      |9023             |293975.23 |168                |5           |
|0          |9574      |8821             |307789.53 |175                |6           |
|0          |3656      |8594             |313540.01 |167                |7           |
|0          |11568     |8415             |274625.73 |154                |8           |
|0          |11584     |8331             |2

[Stage 10:==========================================>              (9 + 3) / 12]

+----------+-----------------+-----------+
|ss_item_sk|quantites_vendues|ca_ttc     |
+----------+-----------------+-----------+
|10000     |454954           |16108841.77|
|10003     |439086           |15614283.97|
|10001     |420138           |15240549.34|
|10002     |377027           |13397798.56|
|5618      |95677            |3355977.13 |
|17502     |94625            |3482713.49 |
|673       |92967            |3187900.02 |
|11552     |92446            |3250624.08 |
|16513     |91710            |3297977.63 |
|689       |91464            |3247954.55 |
+----------+-----------------+-----------+



In [3]:
spark.stop()
print("Session Spark fermée.")

Session Spark fermée.
